In [25]:
import pandas as pd
import glob
import os
import re

In [2]:
#gather all absolute file paths in raw data folder and place them in a list
def get_files():
    raw_data_path = os.path.abspath(os.path.join(os.getcwd(),".."))
    pattern = os.path.join(raw_data_path,"data","raw","*.xlsx")
    file_lst = glob.glob(pattern)
    return file_lst

In [15]:
#clean and filter data to acceptable parameters
def clean_filter(quarter_df_lst):
    #strip and force lowercase on all, force single "_" between words 
    for q in quarter_df_lst:
        q.columns = q.columns.str.lower()
        q.columns = q.columns.str.strip()
        q.columns = q.columns.str.replace('\\s+', '_', regex=True)
    #filter out any sheets with less than 5 rows
    length_check = [v for v in quarter_df_lst if len(v) > 5]
    #filter out sheets without expected cols
    required_cols = {"start","details","status_text","booking_source","player_count","tee_sheet","date_cancelled"}
    col_check = [v for v in length_check if required_cols.issubset(set(v.columns))]
    #filter out sheets with more than 10% NA values on the start time
    valid_start_time = [v for v in col_check if v["start"].isna().mean() < 0.10]
    final = valid_start_time
    return final
    
#for each file, collect each tab into a dict w/ tab name as key, df of data as value. concat all tabs into a full year df
year_dfs = []
file_lst = get_files()
for file in file_lst:
    quarter_dict = pd.read_excel(file,sheet_name=None)
    quarters = list(quarter_dict.values())
    quarters = clean_filter(quarters)
    if quarters:
        year = pd.concat(quarters)
        year_dfs.append(year)
    else: 
        print(f"'{os.path.basename(file)}': no valid data")

#concat all year dfs into one master df
master_raw = pd.concat(year_dfs,ignore_index=True)
display(master_raw)

,start,details,status_text,booking_source,player_count,tee_sheet,date_cancelled
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST...,checked in,online,2,Bethpage Early AM 9 Holes Blue,NaT
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 ES...,checked in,online,1,Bethpage Early AM 9 Holes Blue,NaT
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 ES...,checked in,online,2,Bethpage Early AM 9 Holes Blue,NaT
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 E...,checked in,online,4,Bethpage Early AM 9 Holes Blue,NaT
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 ES...,checked in,online,1,Bethpage Early AM 9 Holes Blue,NaT
...,...,...,...,...,...,...,...
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 E...,NaN,online,1,Bethpage Blue Course,NaT
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 ...,NaN,online,1,Bethpage Blue Course,NaT
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 E...,NaN,online,4,Bethpage Blue Course,NaT
616030,2025-10-21 14:00:00,Reserved using online booking @ 8:55pm 10/14 EDT,deleted,online,4,Bethpage Blue Course,2025-10-14 21:21:01


In [11]:
print(master_raw.dtypes)

start             object
details           object
status_text       object
booking_source    object
player_count       int64
tee_sheet         object
date_cancelled    object
dtype: object


In [24]:
#basic cleaning, convert start to datetime, drop booking_source (provides no meaningful input here)
master_df = master_raw.copy()
master_df["start"] = pd.to_datetime(master_df["start"])
master_df.drop(columns=["booking_source"],inplace=True)
display(master_df)

,start,details,status_text,player_count,tee_sheet,date_cancelled
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST...,checked in,2,Bethpage Early AM 9 Holes Blue,NaT
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 ES...,checked in,1,Bethpage Early AM 9 Holes Blue,NaT
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 ES...,checked in,2,Bethpage Early AM 9 Holes Blue,NaT
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 E...,checked in,4,Bethpage Early AM 9 Holes Blue,NaT
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 ES...,checked in,1,Bethpage Early AM 9 Holes Blue,NaT
...,...,...,...,...,...,...
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 E...,NaN,1,Bethpage Blue Course,NaT
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 ...,NaN,1,Bethpage Blue Course,NaT
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 E...,NaN,4,Bethpage Blue Course,NaT
616030,2025-10-21 14:00:00,Reserved using online booking @ 8:55pm 10/14 EDT,deleted,4,Bethpage Blue Course,2025-10-14 21:21:01


In [26]:
#clean time, date, and cost out of details using regex
master_df[["time","date","cost_per_group"]] = master_df["details"].str.extract(r"@\s((?:\d{1,2}:\d{2})(?:am|pm|AM|PM|Am|Pm))\s(\d{1,2}\/\d{1,2})(?:.*?(\$\d+\.\d{2}))?")
# pull year from start, adjust booking time to previous year if booking in dec and tee time in jand
display(master_df)

,start,details,status_text,player_count,tee_sheet,date_cancelled,time,date,cost_per_group
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST...,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,7:00pm,3/7,$124.00
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 ES...,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,9:35pm,3/13,$31.00
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 ES...,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,3:32pm,3/13,$62.00
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 E...,checked in,4,Bethpage Early AM 9 Holes Blue,NaT,11:19am,3/13,$124.00
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 ES...,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,5:42pm,3/13,$93.00
...,...,...,...,...,...,...,...,...,...
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 E...,NaN,1,Bethpage Blue Course,NaT,8:00pm,10/14,$15.00
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 ...,NaN,1,Bethpage Blue Course,NaT,12:32pm,10/15,$23.00
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 E...,NaN,4,Bethpage Blue Course,NaT,7:02pm,10/14,$92.00
616030,2025-10-21 14:00:00,Reserved using online booking @ 8:55pm 10/14 EDT,deleted,4,Bethpage Blue Course,2025-10-14 21:21:01,8:55pm,10/14,NaN
